# Feature selection by category: Promotional

Finds the strongest **promotional** features, personal (F2F and account calls, speaker programs, iRep emails, messages, MSL, samples) and non-personal (HQ email, digital paid media, high intent signals, DTC, OE), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with all 28 cohorts. Reads only this category's 11 source tables and fills empty values with the Combine rules.

Same method as every category notebook: the category config (cell 3) is the only difference; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4). **Part 1:** writer definition (`TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0, empty = 0), writers / non-writers data sets, mean / median analysis (`DISCRIMINATION` = \|2 x AUC - 1\|). **Part 2:** NBA 3.1 clean-up (stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model per group on all 28 cohorts, SHAP, best features. Outputs: `feature_selection_by_category/<RUN_NAME>/promotional/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "promotional"
CATEGORY_LABEL = "Promotional: personal (field activity) and non-personal (HQ email, digital, high intent, DTC, OE)"
ADD_CALENDAR   = False

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

# Personal: field_core_account holds every feature of field_core and field_account_call, so only it is used (as in Combine)
FA = NO + "field_activity_nba_3_1/kerendia_nba_3_1_hcp_model_"
SOURCE_TABLES = [
    _t("field_core_account", "Personal: F2F calls + account calls", FA + "field_core_account_features.parquet", "parquet", "COHORT"),
    _t("field_events", "Personal: speaker programs", FA + "field_events_features.parquet", "parquet", "COHORT_ID"),
    _t("field_irep", "Personal: iRep emails", FA + "field_irep_features.parquet", "parquet", "COHORT_ID"),
    _t("field_message", "Personal: messages", FA + "field_message_features.parquet", "parquet", "COHORT_ID"),
    _t("field_msl", "Personal: MSL", FA + "field_msl_features.parquet", "parquet", "COHORT_ID"),
    _t("field_sample", "Personal: samples", FA + "field_sample_features.parquet", "parquet", "COHORT_ID"),
    # Non-personal
    _t("hq_email", "Non-personal: HQ email", NO + "hq_email_nba_3_1/kerendia_nba_3_1_hcp_model_hq_email_features.parquet", "parquet", "COHORT"),
    _t("digital_paid_media", "Non-personal: digital paid media", NO + "digital_engagement_nba_3_1/kerendia_nba_3_1_hcp_model_digital_paid_media_features.parquet", "parquet", "COHORT"),
    _t("high_intent", "Non-personal: high intent signals", NO + "high_intent_signals_nba_3_1/kerendia_nba_3_1_hcp_model_high_intent_features.parquet", "parquet", "COHORT"),
    _t("dtc", "Non-personal: DTC", NO + "dtc_nba_3_1/kerendia_nba_3_1_hcp_model_dtc_features.parquet", "parquet", "COHORT"),
    _t("oe", "Non-personal: OE", NO + "oe_nba_3_1/kerendia_nba_3_1_oe_promotional_features.parquet", "parquet", "COHORT"),
]

In [0]:
%run ./00_category_feature_selection_core